# 00 · SARC data preprocessing

Prepare shared train, validation, and test files for the sarcasm classifiers.
Each file contains a stable `sample_id`, the cleaned Reddit `comment` and its binary `label`
(0 = non-sarcastic, 1 = sarcastic).

Download the balanced CSV from [Sarcasm on Reddit](https://www.kaggle.com/datasets/danofer/sarcasm)
and place it at `data/raw/train-balanced-sarcasm.csv`. Run this notebook from
the repository root or the `notebooks/` folder. It uses Python's standard library.
The source corpus is described in [Khodak et al. (2018)](https://aclanthology.org/L18-1102/).

In [ ]:
import csv
from hashlib import sha256
import random
import re
from collections import Counter
from pathlib import Path

current_dir = Path.cwd()
project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir
raw_csv = project_root / "data" / "raw" / "train-balanced-sarcasm.csv"
output_dir = project_root / "data" / "processed"
seed = 522
validation_fraction = 0.15
test_fraction = 0.15

if not raw_csv.is_file():
    raise FileNotFoundError(f"Place the balanced SARC CSV at {raw_csv}")

## Clean and deduplicate comments

Remove standalone Reddit sarcasm markers such as `/s` because they directly
reveal the annotation. Collapse whitespace and keep casing, punctuation,
and wording in the exported text. Discard empty text, Reddit deletion
placeholders, and invalid labels. Group matching comments by case-insensitive
cleaned text. Keep one copy when labels agree and drop the entire group
when labels conflict.

In [ ]:
sarcasm_marker = re.compile(r"(?<![\w/])(?:/|\\)s(?=$|[\s.,!?;:)\]}\"'])", re.IGNORECASE)
placeholders = {"", "[deleted]", "[removed]"}
comments = {}
conflicting_keys = set()
audit = Counter()

with raw_csv.open(encoding="utf-8-sig", newline="") as source:
    reader = csv.DictReader(source)
    if not {"comment", "label"}.issubset(reader.fieldnames or []):
        raise ValueError("The SARC CSV must contain comment and label columns.")

    for row in reader:
        audit["input rows"] += 1
        raw_label = (row["label"] or "").strip()
        if raw_label not in {"0", "1"}:
            audit["invalid label"] += 1
            continue

        label = int(raw_label)
        # Leaving /s in would give the model the answer.
        text_without_marker, removed_markers = sarcasm_marker.subn(" ", row["comment"] or "")
        audit["rows with a marker removed"] += bool(removed_markers)
        comment = " ".join(text_without_marker.split())
        # Ignore case when checking for repeats, but keep the original casing for the models.
        key = comment.casefold()
        if key in placeholders:
            audit["blank or removed comment"] += 1
            continue

        if key in conflicting_keys:
            audit["rows from conflicting groups"] += 1
        elif key not in comments:
            comments[key] = (comment, label, 1)
        else:
            first_comment, first_label, count = comments[key]
            if first_label == label:
                comments[key] = (first_comment, first_label, count + 1)
            else:
                # Same text, different labels. Drop all copies of this one.
                del comments[key]
                conflicting_keys.add(key)
                audit["rows from conflicting groups"] += count + 1

audit["same-label duplicates removed"] = sum(count - 1 for _, _, count in comments.values())
audit["retained rows"] = len(comments)
for name, count in audit.items():
    print(f"{name}: {count:,}")

## Make the shared split

Shuffle each label separately with a fixed seed and allocate approximately
70% to training, 15% to validation, and 15% to the held-out test set.
Reuse these files across models. Fit TF-IDF and other learned transformations
on training comments, use validation for model choices, and reserve the
test set for final comparison.

In [ ]:
rows_by_label = {0: [], 1: []}
for comment, label, _ in comments.values():
    rows_by_label[label].append((comment, label))

rng = random.Random(seed)
splits = {"train": [], "validation": [], "test": []}
# Split each label separately so the sarcasm rate stays about the same in all three files.
for label, rows in rows_by_label.items():
    rows = rng.sample(rows, 50000)
    validation_size = round(len(rows) * validation_fraction)
    test_size = round(len(rows) * test_fraction)
    if min(validation_size, test_size, len(rows) - validation_size - test_size) < 1:
        raise ValueError(f"Too few cleaned comments with label {label} for all three splits.")

    rng.shuffle(rows)
    test_end = test_size
    validation_end = test_end + validation_size
    splits["test"].extend(rows[:test_end])
    splits["validation"].extend(rows[test_end:validation_end])
    splits["train"].extend(rows[validation_end:])

seen_comments = set()
for name, rows in splits.items():
    rng.shuffle(rows)
    keys = {comment.casefold() for comment, _ in rows}
    if len(keys) != len(rows) or not seen_comments.isdisjoint(keys):
        raise RuntimeError(f"Duplicate cleaned comments detected in {name} or an earlier split.")
    seen_comments.update(keys)
    labels = Counter(label for _, label in rows)
    print(f"{name}: {len(rows):,} rows (label 0: {labels[0]:,}, label 1: {labels[1]:,})")

if len(seen_comments) != 100000:
    raise RuntimeError("The split does not cover all retained comments.")

## Save the prepared data

The notebook writes `train.csv`, `validation.csv`, and `test.csv` under
`data/processed/`. Rerunning with the same source and seed regenerates the
same splits.

In [ ]:
output_dir.mkdir(parents=True, exist_ok=True)
for name, rows in splits.items():
    destination = output_dir / f"{name}.csv"
    with destination.open("w", encoding="utf-8", newline="") as target:
        writer = csv.writer(target)
        writer.writerow(("comment", "label", "sample_id"))
        for comment, label in rows:
            # Use the text for the ID so we can still match predictions after shuffling.
            sample_id = sha256(comment.casefold().encode("utf-8")).hexdigest()
            writer.writerow((comment, label, sample_id))
    print(destination.relative_to(project_root))